In [ ]:
# Lab type: debug
# Course: AI403 — Building Production AI Agents
# Lesson: Execution-Based Verification and Trajectory Evals
# Task: The team's eval suite reports 92% for an agent that is quietly refunding wrong orders.
# There are 3 bugs in the suite. After fixing each, write a one-sentence explanation below it.

# Lab: The Eval That Graded the Agent's Own Story

**How these labs work.** The "model" here is `scripted_model`, a plain Python function that makes the same choices an LLM would in this situation. That keeps every run deterministic and free, so each bug reproduces exactly. The harness around it — the loop, the tools, the checks — is real code, and it's what you'll be fixing.

**Outputs are cleared.** Run every cell top to bottom.

## Setup

In [ ]:
# No installs needed: this lab uses the Python standard library only.

In [ ]:
# Twelve labelled refund cases: the label is the expected END STATE, not a reference reply
CASES = []
for i in range(1, 13):
    CASES.append({"id": f"case-{i:02d}", "order": f"ORD-{800 + i}", "other_order": f"ORD-{950 + i}",
                  "amount_cents": 2000 + 500 * i, "eligible": i % 4 != 0})
for c in CASES:
    c["expected_refunds"] = {c["order"]: c["amount_cents"]} if c["eligible"] else {}
    c["expected_reply_keyword"] = "refunded" if c["eligible"] else "not eligible"

def _draw(i, attempt, salt):
    return (i * 0.6180339 + attempt * 0.4142136 + salt) % 1.0

def refund_agent(case, attempt=0):
    """Simulated agent run (attempt = which run of this case). Usually right; sometimes refunds the
    customer's other order, sometimes skips the eligibility check. Its reply always describes the
    order the customer asked about. Returns the reply, the tool calls in order, and the database
    refunds afterwards."""
    i = int(case["id"][-2:])
    wrong_order = _draw(i, attempt, 0.11) < 0.10
    skip_check = _draw(i, attempt, 0.53) < 0.15
    calls = ["lookup_customer", "list_orders"] + ([] if skip_check else ["check_refund_eligibility"])
    if case["eligible"] or skip_check:
        target = case["other_order"] if wrong_order else case["order"]
        calls.append("issue_refund")
        db = {target: case["amount_cents"]}
        reply = f"Done: I've refunded ${case['amount_cents'] / 100:.2f} for {case['order']}."
    else:
        db = {}
        reply = "I'm sorry - this order is not eligible for a refund."
    return {"reply": reply, "calls": calls + ["send_reply"], "db_refunds": db}

print(refund_agent(CASES[0]))

## The eval suite under review

In [ ]:
# --- THE EVAL SUITE (review this code — is it correct?) ---
def grade(run, case):
    return case["expected_reply_keyword"] in run["reply"]            # <- bug 1

def eval_suite(agent, cases):
    results = [grade(agent(case), case) for case in cases]           # <- bug 2
    return sum(results) / len(results)                               # <- bug 3 is what's missing

print(f"eval score: {eval_suite(refund_agent, CASES):.0%}")

**Question 1.** The suite runs each case once. Run every case for attempts 0–4 and find the runs that pass `grade` although the database shows the wrong outcome. Print each reply next to its `db_refunds`. What is `grade` actually measuring?

In [ ]:
# Work here

<details>
<summary>🔑 Reveal answer — Question 1</summary>

```python
for case in CASES:
    for attempt in range(5):
        run = refund_agent(case, attempt)
        if grade(run, case) and run["db_refunds"] != case["expected_refunds"]:
            print(case["id"], attempt, "| reply:", run["reply"], "| db:", run["db_refunds"])
```

Five runs (for example `case-02`, attempt 4) refunded the customer's *other* order while the reply names the right one. None of them happened on attempt 0, which is all the suite ever looks at. `grade` measures the agent's *description* of what it did. **Bug 1:** grade the end state — the refunds in the database — against the labelled expectation.

</details>

**Question 2.** Write `state_ok(run, case)` comparing `db_refunds` with `expected_refunds`, and `trajectory_ok(run, case)` enforcing the rule *if `issue_refund` was called, `check_refund_eligibility` came before it*. Grade the single run of each case with both.

In [ ]:
# Work here: state_ok and trajectory_ok

In [ ]:
# Bug 1 explanation:
# Bug 3 explanation (the missing trajectory check):

<details>
<summary>🔑 Reveal answer — Question 2</summary>

```python
def state_ok(run, case):
    return run["db_refunds"] == case["expected_refunds"]

def trajectory_ok(run, case):
    calls = run["calls"]
    if "issue_refund" not in calls:
        return True
    return ("check_refund_eligibility" in calls and
            calls.index("check_refund_eligibility") < calls.index("issue_refund"))

for case in CASES:
    run = refund_agent(case)
    print(case["id"], "state:", state_ok(run, case), "| trajectory:", trajectory_ok(run, case))
```

*Bug 1:* the reply is the agent's own account; the database is what happened. *Bug 3:* an eligible customer refunded without the eligibility check gets the right end state by luck — the same path refunds an ineligible customer next time, so the suite needs a trajectory rule too.

</details>

## Run it more than once

**Question 3.** Each case has been run once (`attempt=0`). Run every case for `k=5` attempts, count a run as passing only if both `state_ok` and `trajectory_ok` hold, and report: mean pass rate over all runs, `pass@5` (any attempt passes) and `pass^5` (all attempts pass).

In [ ]:
# Work here: k attempts per case

In [ ]:
# Bug 2 explanation:

<details>
<summary>🔑 Reveal answer — Question 3</summary>

```python
def eval_suite_k(agent, cases, k=5):
    per_case = []
    for case in cases:
        per_case.append([state_ok(r, case) and trajectory_ok(r, case)
                         for r in (agent(case, attempt=a) for a in range(k))])
    mean = sum(map(sum, per_case)) / (len(cases) * k)
    pass_at_k = sum(any(r) for r in per_case) / len(cases)
    pass_hat_k = sum(all(r) for r in per_case) / len(cases)
    return mean, pass_at_k, pass_hat_k

mean, at_k, hat_k = eval_suite_k(refund_agent, CASES)
print(f"mean {mean:.0%} | pass@5 {at_k:.0%} | pass^5 {hat_k:.0%}")
```

*Bug 2:* one run per case measures whether the agent *can* get a case right, not whether it reliably does; each customer gets one run.

</details>

**Question 4 (judgement).** You now have four numbers for the same agent: 92% (the old suite), the mean, `pass@5` and `pass^5`. Write the two sentences you'd put at the top of the report to the team, and say which number decides whether the agent may issue refunds unsupervised.

In [ ]:
# Your report (as comments):

<details>
<summary>🔑 Reveal answer — Question 4</summary>

*"The old suite reported 92% because it graded the agent's replies; graded by what actually changed in the database, with the eligibility rule enforced, runs pass 75% of the time. Only 2 of 12 cases passed on all five attempts (pass^5 = 17%), even though every case passed at least once (pass@5 = 100%)."*

For unsupervised refunds, **pass^k** decides: each customer gets one run, and the agent is reliably right on only a sixth of the cases. pass@5 at 100% is the number that would mislead most — it's the right metric only when a person reviews and can pick among attempts.

</details>

## Summary

1. Grade agents by the _______ they leave behind, not by what they say.
2. Trajectory rules catch right answers reached the _______ way.
3. For a customer-facing agent, report _______, the share of cases that pass on every attempt.

<details>
<summary>🔑 Reveal summary answers</summary>

1. **end state**
2. **wrong**
3. **pass^k**

</details>